# Comparison of Two ROS3 Backends when Accessing Single-Shot HDF5 files in AWS S3

The forthcoming HDF5 library version 2.0 (as of writing this) replaced the code for working with S3-compatible web object stores in its Read Only S3 (ROS3) file driver with the Amazon's C S3 [library](https://github.com/awslabs/aws-c-s3). This notebook compares benchmark results between the latest current library release 1.14.6 and a development version of future 2.0 release. The goal is to assess whether there are significant performance differences between the old and new code in the ROS3 driver.

The same single-shot HDF5 files with C-Mod data in S3 were used for the benchmarks with the new ROS3 code as with the "old" ROS3 from the 1.14.6 library release.

## TL;DR Conclusions

* New ROS3 driver never reported errors when accessing the HDF5 files while the old ROS3 had occasionally done so. This is likely the consequence of the Amazon's S3 library and its intelligent handling of non-fatal failed S3 requests.
* The new ROS3 benchmarks are comparable or better than the old ones. The biggest change is when reading all signals from all the files where a ~30% improvement was observed for the new ROS3. This case also did not have a performance degradation when the number of Dask workers significantly increases compared to the available CPUs (contention for resources case).

The overall conclusion is the new ROS3 does not degrade performance compared to the old ROS3 for the HDF5 files in this project.

---

## Benchmark Data Analysis

In [1]:
import pandas as pd
import hvplot.pandas  # noqa: F401
import holoviews as hv

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

hv.extension("bokeh")

Read benchmark data for the new ROS3 backend:

In [3]:
s3_data_new = pd.read_csv("./bench.hsdsv1.csv")
s3_data_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 5830 entries, 0 to 5829
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   worker#              5830 non-null   int64  
 1   obj-id               5830 non-null   str    
 2   open+read-data-time  5830 non-null   float64
 3   wrkr-num-objs        5830 non-null   int64  
 4   mean-obj-time        5830 non-null   float64
 5   num-dsets            5830 non-null   int64  
 6   mean-dset-time       5830 non-null   float64
 7   num-workers          5830 non-null   int64  
 8   obj-type             5830 non-null   str    
 9   tot-num-obj          5830 non-null   int64  
 10  total-runtime        5830 non-null   float64
dtypes: float64(4), int64(5), str(2)
memory usage: 582.5 KB


Read benchmark data for the old ROS3 backend:

In [4]:
s3_data_old = pd.read_csv("./bench.hsdsv09.csv")
s3_data_old.info()

<class 'pandas.DataFrame'>
RangeIndex: 5830 entries, 0 to 5829
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   worker#              5830 non-null   int64  
 1   obj-id               5830 non-null   str    
 2   open+read-data-time  5830 non-null   float64
 3   wrkr-num-objs        5830 non-null   int64  
 4   mean-obj-time        5830 non-null   float64
 5   num-dsets            5830 non-null   int64  
 6   mean-dset-time       5830 non-null   float64
 7   num-workers          5830 non-null   int64  
 8   obj-type             5830 non-null   str    
 9   tot-num-obj          5830 non-null   int64  
 10  total-runtime        5830 non-null   float64
dtypes: float64(4), int64(5), str(2)
memory usage: 582.5 KB


Page cache sizes used in the benchmarks:

These were the benchmark parameter combinations:

In [6]:
s3_data_old[["obj-type", "num-workers"]].drop_duplicates()

,obj-type,num-workers
0,shots,1
1,shots,2
3,shots,4
7,signals,8
2000,shots,8
2008,signals,16
5814,shots,16


In [7]:
s3_data_new[["obj-type",  "num-workers"]].drop_duplicates()

,obj-type,num-workers
0,shots,1
1,shots,2
3,shots,4
7,signals,8
2000,shots,8
2008,signals,16
5814,shots,16


Column `obj-type` describes data access type during one benchmark run:

* `obj-type = shots` means all signals from one shot file were read.
* `obj-type = signals` means all signals from all the files were read, one at a time.


Since the two ways of reading data by `shots` and `signals` are so different they cannot be compared to each other. Separate them into different DataFrames.

In [8]:
s3_old_shots = s3_data_old[s3_data_old["obj-type"] == "shots"]
s3_new_shots = s3_data_new[s3_data_new["obj-type"] == "shots"]
s3_old_signals = s3_data_old[s3_data_old["obj-type"] == "signals"]
s3_new_signals = s3_data_new[s3_data_new["obj-type"] == "signals"]

In [9]:
s3_new_shots.head()

,worker#,obj-id,open+read-data-time,wrkr-num-objs,mean-obj-time,num-dsets,mean-dset-time,num-workers,obj-type,tot-num-obj,total-runtime
0,1,1160926028,55.792627,1,55.792627,776,0.071898,1,shots,100,55.813563
1,1,1160930031,38.919221,1,38.919221,373,0.104341,2,shots,100,38.939917
2,2,1160930031,22.222220,1,22.222220,432,0.051440,2,shots,100,38.939917
3,1,1160926028,1.163064,1,1.163064,162,0.007179,4,shots,100,1.871618
4,2,1160926028,1.846029,1,1.846029,213,0.008667,4,shots,100,1.871618


In [10]:
s3_new_signals.head()

,worker#,obj-id,open+read-data-time,wrkr-num-objs,mean-obj-time,num-dsets,mean-dset-time,num-workers,obj-type,tot-num-obj,total-runtime
7,1,p_to_v_expr,8.051642,13,0.619357,13,0.619357,8,signals,250,1616.973469
8,2,p_to_v_expr,8.754190,13,0.673399,13,0.673399,8,signals,250,1616.973469
9,3,p_to_v_expr,8.254301,13,0.634946,13,0.634946,8,signals,250,1616.973469
10,4,p_to_v_expr,7.867847,13,0.605219,13,0.605219,8,signals,250,1616.973469
11,5,p_to_v_expr,8.315548,13,0.639658,13,0.639658,8,signals,250,1616.973469


### Total Runtime and Peformance

Total benchmark runtime in the `tot-runtime` column is the elapsed time of the entire benchmark as measured by the main process. The total runtime encompasses:
1. Dividing data access plan across Dask workers and their intialization.
1. All Dask workers completing their jobs.
1. Collecting Dask worker benchmark data.

Below are four DataFrames with total runtimes separated for the signal and shot benchmarks. Their runtimes are so different that there is no point comparing them together. The new DataFrames include several original columns plus a new column `norm-tot-runtime`. This column holds computed performance ratios to the _baseline_ benchmark. The baseline benchmark is one of the available benchmarks selected because it represents the most common set of libhdf5 settings, compute resources, and data access. The baseline benchmarks are:

* S3 files:
    * Reading all signals for a shot: 1 Dask worker, 264 MB file page cache
    * Reading all signals for all the shots: 8 Dask worker, 264 MB file page cache

In [12]:
old_shots_runtime = s3_old_shots[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
old_shots_runtime["where"] = "Local"
old_shots_runtime["norm-tot-runtime"] = (
    old_shots_runtime.loc[0, "total-runtime"] / old_shots_runtime["total-runtime"]
)

old_signals_runtime = s3_old_signals[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
old_signals_runtime["where"] = "Local"
old_signals_runtime["norm-tot-runtime"] = (
    old_signals_runtime.loc[0, "total-runtime"] / old_signals_runtime["total-runtime"]
)

new_shots_runtime = s3_new_shots[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
new_shots_runtime["where"] = "S3"
new_shots_runtime["norm-tot-runtime"] = (
    new_shots_runtime.loc[0, "total-runtime"] / new_shots_runtime["total-runtime"]
)

new_signals_runtime = s3_new_signals[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
new_signals_runtime["where"] = "S3"
new_signals_runtime["norm-tot-runtime"] = (
    new_signals_runtime.loc[0, "total-runtime"] / new_signals_runtime["total-runtime"]
)

### Reading All Data from a Single Shot File

Plots of performance ratio and runtime when reading all data from a single S3 shot file:

In [14]:
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
(
    old_shots_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * old_shots_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
    * hv.HLine(1).opts(line_width=0.7, color="pink")
    * new_shots_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * new_shots_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
).options(
    legend_position="top_right",
    title="Shot File: Old (blue) vs New (red) ROS3",
    xlabel="Number of Dask workers",
    ylabel="Performance ratio (>1 better)",
    xlim=(0, old_shots_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
) + (
    old_shots_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * old_shots_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
    * new_shots_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * new_shots_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
).options(
    legend_position="bottom_right",
    title="Shot File: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Total runtime / [s]",
    xlim=(0, old_shots_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    show_grid=True,
    height=400,
    width=500,
)

:Layout
   .Overlay.I  :Overlay
      .Curve.I    :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.I  :Scatter   [num-workers]   (norm-tot-runtime)
      .HLine.I    :HLine   [x,y]
      .Curve.II   :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.II :Scatter   [num-workers]   (norm-tot-runtime)
   .Overlay.II :Overlay
      .Curve.I    :Curve   [num-workers]   (total-runtime)
      .Scatter.I  :Scatter   [num-workers]   (total-runtime)
      .Curve.II   :Curve   [num-workers]   (total-runtime)
      .Scatter.II :Scatter   [num-workers]   (total-runtime)

### Reading All Signals from All Shot Files

Plots of performance ratio and runtime when reading all signals from all the shot files in S3:

In [15]:
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
(
    old_signals_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * old_signals_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
    * hv.HLine(1).opts(line_width=0.7, color="pink")
    * new_signals_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * new_signals_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
).options(
    legend_position="top_right",
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Performance ratio (>1 better)",
    xlim=(0, old_signals_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
) + (
    old_signals_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * old_signals_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
    * new_signals_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * new_signals_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
).options(
    legend_position="bottom_right",
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Total runtime / [s]",
    xlim=(0, old_signals_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    show_grid=True,
    height=400,
    width=500,
)

:Layout
   .Overlay.I  :Overlay
      .Curve.I    :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.I  :Scatter   [num-workers]   (norm-tot-runtime)
      .HLine.I    :HLine   [x,y]
      .Curve.II   :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.II :Scatter   [num-workers]   (norm-tot-runtime)
   .Overlay.II :Overlay
      .Curve.I    :Curve   [num-workers]   (total-runtime)
      .Scatter.I  :Scatter   [num-workers]   (total-runtime)
      .Curve.II   :Curve   [num-workers]   (total-runtime)
      .Scatter.II :Scatter   [num-workers]   (total-runtime)

### Display Some Worker Mean Runtimes

The `mean-obj-time` column holds mean read times of _objects_ in a shot file. Which _object_ is read depends on the `obj-type` column, with the values `shots` or `signals`. The `s3_signals` DataFrame 

In [17]:
(
    s3_old_signals.hvplot.box(
        y="mean-obj-time",
        by=["num-workers"],
    )
    * s3_new_signals.hvplot.box(
        y="mean-obj-time",
        by=["num-workers"],
    )
).options(
    # ylim=(8, 11),
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    height=400,
    show_legend=False,
    xlabel="File page buffer size, Number of Dask workers",
    ylabel="Worker mean signal read time / [seconds]",
    show_grid=True,
)

:Overlay
   .BoxWhisker.I  :BoxWhisker   [num-workers]   (mean-obj-time)
   .BoxWhisker.II :BoxWhisker   [num-workers]   (mean-obj-time)

In [18]:
s3_new_signals.groupby(["num-workers"])["mean-obj-time"].describe()

,count,mean,std,min,25%,50%,75%,max
num-workers,,,,,,,,
8,1993.0,0.534262,0.278311,0.220715,0.325284,0.533987,0.645374,3.302288
16,3806.0,0.685725,0.286470,0.231105,0.527410,0.655353,0.801454,4.310197


In [19]:
s3_old_signals.groupby(["num-workers"])["mean-obj-time"].describe()

,count,mean,std,min,25%,50%,75%,max
num-workers,,,,,,,,
8,1993.0,1.644562,0.483214,0.867062,1.364350,1.497909,1.755008,5.432870
16,3806.0,3.984667,2.099014,0.978687,2.769555,3.485612,4.494150,39.513084


---